# Greedy decoding for a seq2seq model

Write batched greedy decoding for an encoder-decoder model, the loop of lesson u17-l02, without any library `generate`.

```python
def greedy_decode(model, src, src_mask, start_id, eos_id, pad_id, max_len):
    # Your code
    return out
```

The grader passes a `model` object with two methods. You do not know what is inside it.

- `model.encode(src, src_mask)` returns `memory`, a float tensor of shape `(B, S, d)`.
- `model.decode(tgt, memory, src_mask)` takes `tgt`, a long tensor of shape `(B, T)` holding the start token and the tokens generated so far, and returns logits of shape `(B, T, V)`. `logits[:, t]` scores the token that comes after `tgt[:, :t + 1]`.

Inputs:

- `src`: long tensor `(B, S)` of source token ids, padded on the right; `src_mask`: tensor `(B, S)`, 1 (or `True`) for real tokens and 0 for padding. Pass `src_mask` to both methods unchanged.
- `start_id`, `eos_id`, `pad_id`: the start, end and padding token ids (`start_id` may equal `pad_id`, as in T5). `max_len`: the most tokens to generate after the start token, the end token included.

Return `out`, a long tensor of shape `(B, L)`:

- `out[:, 0]` is `start_id`. At each step every row gets the token with the highest logit at the last position.
- A row is finished once it has produced `eos_id`; from then on it gets `pad_id`.
- Stop after the step in which the last row finishes, or after `max_len` steps. So `out` has one column more than the number of steps, and a row that never produces `eos_id` has `max_len` generated tokens.

The result must equal decoding each row on its own and padding the rows to a common length.

**Example.** The example's model writes the real source tokens in reverse order, then the end token.

```python
src = torch.tensor([[5, 6, 7], [8, 9, 0]])
src_mask = torch.tensor([[1, 1, 1], [1, 1, 0]])
greedy_decode(model, src, src_mask, start_id=1, eos_id=2, pad_id=0, max_len=10)
# tensor([[1, 7, 6, 5, 2],
#         [1, 9, 8, 2, 0]])
```

Row 1 finishes one step before row 0, so it gets one padding token. The hidden tests use small random networks, several sets of token ids, rows that are cut off by `max_len`, and a model that never produces the end token.

**Restrictions.** Only `import torch` and `math`. No method or function named `generate`. Call `model.encode` once per call of `greedy_decode`, and `model.decode` once per step for the whole batch. Do not modify `src` or `src_mask`.

**Constraints.** $B \le 64$, $S \le 40$, $V \le 1000$, `max_len` $\le 50$. Time limit 1 second.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile submission.py
import torch


def greedy_decode(model, src, src_mask, start_id, eos_id, pad_id, max_len):
    # Your code
    return out


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import submission as submission
importlib.reload(submission)
# e.g. submission.greedy_decode(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XI132zaS/1dwznsbMlUYyfloT6l7m6RuN/eSJi/OdveerMdSIiRxTZEqQdV2vb6//eY3A5AgJSd2urferS2BwMxgvjAzGObqIEsPxupgO/r64Wb49GCgDta6TmjoavdJndW5xtiPldbppUr1vEyzYqkWZaUSZfSvh/SfWpepzjE/18aUhQOSDw8ZfFZsa21odPSUvqbZYpHNt3l9SSOPaUAXdYXPB0tGEjMSzSvLdJvruEjWTITZztaZMRlhoIdJnpfnOo2z9aasaoCfHNRlNV/xyqReHUzpwywpCp3yw6UudJXU2o3HeVlusG6R5EZjt9rUcZ2tdbmtYzw4HGI0WQrwIt9EdsdA4XgRCdnMLqCP1ok5o3FGY8ptNWfiyypbZkXCbNokVX1Ig3W11df0fZHlzJ+rg1VSFcTDaMMMOeX//UXnG10ZVa+SWm2qcpbrtVpWSYpB2b0KfllU5VrZ5W70wS9hdFqcFi/sdJUZNS+Lmjj+aFtktXn0Lf5898hCpe+bLP3ukcwmIiL1uiahLzKCqsAetdgW85okYBSkkvJg/EBoq5MzTZ/0adFKSokM1S809Eukjv/+4u37N8cnKs9MjS1pAQtgRplVeV4oWvRhW6jgt8xkRFX4XJ0AXK2qLaHVv+nqkhfR1n5Isnxb0coqyYxW+KrOs3pFuknwTLLU6rzKatowa+xLXYyVSS7VOcg9Tww9LYvlQBWAqtLteqNWWZrS9KzYbGsD7okUCsvSebm5bL5AV8Bf+7Wg9ZeKoBYbjN5Tr8o8malX7/9qVFJpZaCwFXhVgJT7Rr09VO+rcqBMqfJynuSASKpj1HpLTJlpMJVYVMJiMnqcZ+DDjPa/IlEukjlpXARMh9EQwk3Ucgv5B4LZEKxcP6xXlU5SZTakp5hVldvlKr8kbckXimghKuZlBUZXGkyvVbYAgfSVTcvuw1oGmHLy/vj4+/iHF68+vvugjoAdGz4t5nlC2CGG4PhirjfQlHB8Wij62dAjmUUKRRzXaUC6mA7U2izdHMJblGBykdoR/LTCDTC3BRIXm+DCrSUn4q1xEoJFeqML2n9WmDop5jq4GMjz6KMuTFmF3nJGq+ttVaiLKCX/OF8FYTTfbOk3izkIZbLmbarXjO64qsrKgyJb9mAVmygxSVUll0R3uw+zSjY6WJb1gJSywG/Sz6PTA9Js8pOnB44yQ7yut5tcB9g5zQ8jXmppYZ7SHDcJsMKBWpweXAHgNQncCC51Zcjv6IuNntekE1fe/GtC11I2z0vzKcoGqqrL/GikHz4bqMR+/MbRS5Z1TiQ3xCamvqSNLvIyAWEYZ5zdB95mlrI/bOlcPt6wHdmAnR7u25t7xNtrEJBAMkPuLatpNdGR50GoyFNAC/1n5/aZjx6eNMnYExZYkxWLBrhVZMg7zy0TiRmWXfhl2YVfvuLpqiKOMSNA3Dq5wJ9kZoKleqjOwzAEbNpo9rtWms4tNYTx7TGVLp/Y0anfkpz8gwrypFrC7WoorLqiP+Po8fI67Mhe/7pN8ltoZdeoZCqILlLW9EHIHCUikrqu7PPTAxZHC2SPungK0s75f1cLXzWY/tjygYjbMSafqw0ESGXcp5iUAdTy5lsY5Iyv6NF/VB3SMIuGOsLITMwmwuLAJ3OjPBihJ5F2yQ79GCVWWMBhFHOsFccdcpbk/iA5i/W+m3TfQe1SGpsi2dBJXv97XXNeFvqOTplQOq28gEry2X4Q0pGYsvV26QnSbE6MROAS+vR0N9ehDgCDVOvN0UeK9DwFswR+JPb1ydsFEHaOEAxFgMnPLjqMJ7EEyUDNvpzvya34LlPELhjfl7I9AdsZjum6AosoESQmmIX7fA0tb/3M3uWNn5jZjyzbrmVjBwPxdqTZhS8rB4cheMFLGa+3ZBQU3QSLYqAekDsly3jw4OwcnxwlEjq+ohNALQrGvECAij1QjENR4JLMKykoZKyW2zUF5XSEqE2ezHXkok4+9WFP5BcnrWUloc3AaAEwTmUmRWc4PIqgT5A8xpKMFJk2bEIs1RTKcEoU/J5tAlnDODoKzv6kUS7rRC4psVHI0xDgZ4sMkSVC5nYv964y9ZUaXT9X52V1hqg+YeVVEB/Fo42/tEwm4lsOI4jPisBohIJE1W25/NxnMXlXP+TeE0lzDknnoaHYm3YABnGse79ZScG75FBA9Y5AVEjBKEFaG2yJ4/C3yVzNCQGWICvmqBvZCgf2CA4ou3g426bkSTHMMTiF5RkSqO2GzEnV59lcAyBxKKMTulxYCyaaHUcNqxBGFhQv4fgGBFqxpWeR+ogHWUXDtJn8PlDgXBBk4LBOO2pVDxFNEoSIsstFzGB05bzNZ1QprfcvphiltuGI3e+RskJUj5SfNrQqGUMVK1hDcOjrHQmR0Hx7ZCH1XNGMcpqzdig5HKizQ8LW9Y9M8qA32NnJ51nBdIIRh8yGQ2+Y2UAZW5DiiL2BHX686W+I7aguS9YbSv7TehwdLq4pyict089t1tUoMSsTmR0DUtsC+fyVZY1dxzkkJXZOxHt+GCVllpQGL0i1lghgJcm8snJiQLCOTxpoVSzJOnXqBNYmOCTJtFwTtxcJqV/cTDzlqs3WJElWxlxh6FQ5fsQIWQBR1hYPMnYaqyxPYXVzVEYoYc5gC5j53yfvfiJaC7YcU6dEIdvq5rJe0cjD18rhUd/aEkecZtV36tsWBz2k72SV/+Sx+rudlJ+MzUv/SbAXdZ7NmiH5QyPRts7ydrhsPv7DlG2RYJPUK3+5yZZF0i4zl6ZXXnBfKjoXZsn8rBkhLS5QLJBag5lntM1CN+6OBs6y+mGuaZoaRc+ikY3pzBh1ojVqdQN2KZRd2MPHOb9NkhGXl6eFwxEtspw02n0NTg+I7LLSyABtpYUi0ejBX4uzAjUcU+aoqJRcASCE2QZSix7YSPEvxx+OyXIsL6L39DeIY5TB4jiMyHFhOUV1m6Qiyk4L4kqEyRFpq67qYEjGQOEDwIQEcF/ti2tB6h6d1b8mY3X8ZHjIlbA3f3vxPyfxyxc//XT8PVFwdXoQ2wpiHGMztDEnT/mqL/TcfqI4Xz7Ny/WGaD09uD4t3rx79z7+6d33xycELiBdiX4oyVbx4W+kuFo+vjCXxbx58IZiyFcEQ76daO/L9xRktt9+lJJlWR1fbEK/xPJRajGfq7LECWV66wA6tl2Tv6FjSzcWy6miA+RXIxDZzOP5Ss/PTGCqOURcJ93z9oPYO6JhOqzAx/JMmLM2S2LMc3KLiAClxsdmY+r7jXQ4GLKlRBxUdZVJWbFzRMkhNJnui2brSlNgx1wiLaH8mgjtxqEnl5SfX3AciqKc3o0PJ5bsMWkLl4At9WO4ScPLbYpcFuJnrnSEv0V5PcZnmk3Jz7Ul0FakcSQQvyJyzGQnvTI1eDSZWkJnSTPO52Ttjhwuh8Zbw8C88RnZQyy1UjsOFaeQkWw3S8lUEIlVEigIYIIg8wNkNAi6sxmq8bSdMy2F+wiKbRFwSdzj+Tqp19vcH1lwdYRcNXwzMf88yc8CCKN3dHthOqaLPktq0M8qmlAWEyMmd7x7gNmSCsXLQcKTEJSvUVqQrxFhI0YicgzWFHmSO0IyB+BrJlYEEY73n42eLKIkTS2OcHdyKxtvXmQ2eQZxA+fk4Wi6Z2ErvGi7SRF171nrJ4n5p9n4A3m9/m5IoCQr5qMtvNP2W4Paw0teYRlJn/8AF/scJGg9Lnis63Bgnwb0lt7EvC9ayvu8K9NfONvp772nEEwHDO1WUH/yPPKnAGbpzbvaP2kfxvaw6uNkw4+SzUYXaeAqPrLo9EBcnrj3/+ryax8WPgVfZsU7e4q92C5fGJxBodRWegui0s57m9RvKWTc0WnxQXckjgjzFNIDiSDRgrrR/bsoQKxEVB6jbBCYF/2jRHrKDjbw8IThfu/yFU4UYowDdXV/oO4LEGdRMMD7hI/y3uV9lGKv99cT99GPksUe8o2jvQV2r7lBnCWFAW/nK3tCcCjYHhBcftYJJbF/57/hc8lG8/Pk0jyUe1XKZCguzBpAUEoLrSzyy4HDSZlLe9QEogVSRLKhFIKOS5XrJaXm5wibFM+Moij0DjMKylRaQigCeJVxfilSCLxD8098OHqnsJBrD19Sw3+qwDtL/6Q6caGnQoThzqpDFBhl+QP4gQn36A5BDv+IhIvS4WiuZFspE+XdzcdiRHG5kXiyW23DdDGynundZsPIKEkv/qwcbKSsnt2oQCIni2EynPrKvWf7d2FBi/VTm2fntrNlHv2yDUuYxMre2SgNw4yh1JVe6YKT2CZ25FX/2v3bnV1/IlNfVBmByS9jjmQDAkGJboxsqhvRvyv0Q6Yz1WZeZZxVqA2pao2GD3sTzRMyd32NhI3MVWOe4cv6boVphrKKy1sjypvpcx3Xs0BHcdw8oJTPBpsATpHmgg/1BfAQCJLUAumndjEfMr9mCzYqPUfJBAUreFmuTVqlo98Ix7yovTMEUbDCYDZfp7XkW2bixoRvSLR3P4Lo//qKsV5jvmP2miLxoMnFyNP2s1xkskm1/G3ShIhuKzfOPJx6KbHlFALwI9VMeezCddJ7GkfBIcrLJDVBwEQ8gtrAJrgUcQB4dGahkhE4Z0fpEyBaYvwJjhtcTTxSu3YhCSOGJlN8R6pn2q9SmBGVtZk0NBEDP5UkAh/+pIU2BTk3pKKNoSOChQmaiVA15dMC39hKpn4461I6F88arjvuoPW9BFctAmYnmkNMINP9U160xJU8F0R0VkYnNconr9855kmRJ5I/gf128vrHF28+vB3YLN1OJXcWc3HX3QJ7vqzboAReHg4dLd3smJtg2mIViTPNKk3GJ8IIiNJ+mOVo1HUGFJUj8/XH12+PP8Qfjl+8GTTU9cIcs9HQnm4pLMJojNqMVHZQneQrE29HXoeXqEZbmCMxea6qh28720VnYTFCoA7wq7+QLEbmmQkec8IwFcXf3RLbkK4i1IBiWQbPE34B44bd2oQtu3RKElYRrXFMxZvJHuGB2auxx63L8ozvFNSVk8c4GqIC7FyXxfIyMbqpEEkNpC2JvXxzPByOWgLutI2bKLYXQ0wmbmLQIVZaQdkg6BPnUWPWO6D3Mcr5FeAlfYZG8U18QAn4k+FwOJ5+sSFzS2K3jsNDfn+HzHF3xe5Gk/YykEfh54Xb8iottQRN0mxHpwtAXPt5+x23sLybTZ4eSNMfrLA5Mpo+wGbfto/w87a39IxveZMtCTQ7y7b72cKWy/JkCjyDax10Qbx1ezhrEDQElKgsxLHjE3z7b0llHI4oq/XaBNw9U3RPBfaq7t4fFygJkREsQr/gRV4Cl5AckgBjL3jm4xjaX23Jb4lSIFrhLoLCba3ncOGes2KrffnN5YAVfzhmIJOn4yn5b74XJmptlVqxezw9sKDdZA8bnqIajLRnX5pi/a1c+mBoGA2v73Inttv48AWnzh89eezl3K5X/iKXZoUwaTkHS0Xgvdu7sePB28USe4iNs7POy2Lp7mX6PlsFWSFdZhzND5BALJJKuZvB0HcCFjlfbvRq2n9www3NCK/39Krc6Sz5V9Hi9xnwfUV6yzOkhdUoOOBVpMVpcNNF7UA93ne82XB26hKyqnFsdz6KmgUSqx4hdQza2PVTUSk7lVlZ5sEOYfIMoOqO5gJc3QHXrHAJyo1HCrc1L5RLeMS7xTHyG1yUjd31ANIdudltDov2Tte+STBWM1SHSHjL3msEHMQkUhuSHnJ9QakknYW6gNBtqUjeCAhCaXJoOtOI5eg4oWybFvAdYHJacBMGJ+IMT5YSvGS+IvGfIxvPcH18XjwXvJLHc6z97AkqVY5YziSkg7pimEssTLCHqHdFbHuq/G9RUUSuMpOwsf5w80XlgC/1bRPSwO8vGthWGMijaZ6n484eW/oiWW/g/qf+7eAHtLMb/Rbb6+T4LxTyOpxvROTlGE0y6I/XcktHz3IlbyzQ4zNdWF1kYOSW+CCuke5jtuYaHs2SS+QzrTfGqZVGrxwE3JmokkUtXrBuGGjbSXBXGcMJxnFgdL4YqN8oMpnFaHMltpQmzjp3DpgT/TyQv/KYuLK7xkdgVUrAcy5Jv2K8rbGndQzJpnrQzIjgwSlXjuoyEPFahQknURQNOJGdOhD3VPByoE4GahSO7c4zNDENUUXZJGnKV+stYVa9hbB6WSPJXZfV5X4CCfRHHM3LWlravEYteO8uwZHZroM0Wx+NQnWrH6Y97KhCrotl7XcMphe4Y5qM3bYfqhH8J7MlkUaej+EEzwZqPL0ZjfoYOhxk9BkfLJQtyCsWLTbDgTgYIrweTu3m1G1/rED8Dt8LbmBikrmKE2BX3x0pOgQIY7RM0OoVjPDtIiKrWm+CdVYcUbrtOiQX2zyPcYuKpR1N3A3J1YiCK1KnHyJiSowOPiLArvl5j1LtZdZA/dzpAfjAjTb7jNys0Q0njTjkWWo04EXqjXVgtcdt0j2cadzKtqwh0nH91WiqgnmyNUku7rcUg4dakVsUiT1v+9Ma+75vxEmSm0dXHUeDNbct8XfMEw/gXkBJcG9QXFLIoVFOZS+UVU4nuEGZ3dCXeAtwFy41PRo9kzE0Oh4dRkP5NssSc0RS6TSgNzrRNF6QCRGJaA01/HaFdDJ13NBxswocLwKfFiJg6UAdLek7KolHXWH3wL0ncMPoKWmLD/Sb4RcB+1uPtrTLqruCa1lrv4Cn3lcwlTC6ae0M9/Du/njtO7VPed8dm5OZdVKsApHThOBMKXocRo9DYu/6Fi6DuEIG/7uuyn+n797Z5u3cuPjuFua8vmhcpw/hke+8b+U609BZP24CPaVgwz3q+v4RNvmVGn1Oo1ac6rNgiCHTaL5dd4hk4O4YuYHcxjWCQo1W24ULXcjtUBrVReepxIpIFHubjD9CK4hbjis4tjxCbeR3RCD+7Mzqrj8dH94FLDrs2RdRc9Q1MdLWoMMOHLqtEEO3l9bMerYgmPzT4xVSH1LmztHxHrmD4SIInDH7aW5PXtleIamRcMcxBtef98y8bCd041G76SIWT9B8FZuC6soscn3DGx3HDV3hDSYHHTwd7fClJSayUHf7nW8w9s8jtvv4HGILdR9i25BHKU1c6QWFKeiZcLzzPSb6IZOqbrwu/m7QloD3K5OLmGK4pu8PZ3HbzyWntGQs0pcb+FtBDjlrG7QJn3iuyXC6c4EwcM56MhvP4AGmLYHNUK+3g52TE7WTASD1i+z6VxDtdjnd7SDz2sh7O+72uVhb7vDe+gV+5yWYEK5p2Dry9b4mMI4fkT1bGyYdxZ0iSQ6v7O1rAyGoroJAq8P9DW4X/JqYSPCmRrVu47sTqYNNaDot/G2wGgTEkwBzQ1YK+Sol04qZh0dhqzi++0bE7b1GMhvIkvYNEl7cbWKY0KyxIJm2zleYXH3ixnqdnOmYs3B7BvshC9mCC+0oIIeYj5q3Bu4Uv0laYzoBEgRqoRJiOcRwpnYipe56MgQCcaK++OceIotcJ2ikLbSCpCAJ/1a2SyBzwGeJJDf7aITl9Y/nEy81+7bZxWfiAZdBcS0UhTTkh3zQdsQI9yCBAq4X8uB/xTcNWeHMmff6wLaQqxLfmd3SkcmLj07mUsJD9u9ONHGQ4a3cm9gIURJ1/vGJwEK9G2Hh/vcvCUfvrTrvNcyx6iBuXtWxb2XSWq/r4Hn7QhrDU2JM3Vea0TrClssvgjfG28FpK2fIsGTqVbPKf/VTVpPOPXuC5hZ4gF1U2ZoycUJ1KLnitm4OCG7W8A6Mm0jovKzbAAhpuw0l0L43oUjzJe5IfLi9d7q5SAo4UGpLhRWdfYu7K4B5mW/XhRpy7Ih3xIQy+08w8BCvt/rucOEtXejOzYfz3dRGXkZkZem9JuyY1X8hp1FSzG9tyy8M8sXIeNeXuKNu8nSgKD3+mmQz+Wag/hMMm+5zHs0C8oj4PxbIR28BR4hHnfJjMBq6jR8d/jGb3GUtYT/k8HA07FS0YTWWUzf7GW+xvOdhK6lhj5NSRomlVuzzc54Y7sOffDqB/NmqALOBhe95sl49IRBx4N+dGQpz6TdRORqRyjq4H58SwMtcj/uqicY7DBEK3zczYBRBsNGnQ972YwH9FODDgT9voJ7Q46HHH1KK0ePOpMeHlqyhTyjpz+iJpfOe26LUotzdl1R7/LKiwS5KoyUsRSV+vrXdnd796y05iYiEpeLH4k7iiPn6gcXPEgj1oIZy8Pdk06iSV3cLfrareO6u2e+jtBe5nh7YOp3VUCaPX6l5ed0sQvckf7je1U8yIYr7YhZ4z9z3bfwZC+rJkHXgcNQ3XW9vT6wSjLzy2bBTPntIChANQ5Z3t9KPFlD+N3XmEG7xWWNslOhr2GLnRkjgaL7D6+8d1VTJGfijOIoYF6zIlm/BjcMnYhbPhB+PHT86rhXxzj7hPxOqH4/C/Xs5ZMeCZFrgMc2NE3e1E2CwR9mIa+neUWbJaJNYOstGfDLY7/g3jIAAAUN/8jVf3xs2IESWvOPnNnsW3yGWWNLB1T1CW1A2LPn2SOhl3HZsP2556OO+4pXX7a0ff3/efUvV8imrmRrc9vM0ucNcocu0zO0W9pqAuy+LST4x/0tHt7GFJ2IJpMYi/yefsgeZBQfZtYinYVshmPOFLnLG3Sulm080mu+pzdPhXc5J+6b8CIbZuTcEKZ5e7kYZn41cfIqgyTYwkjduGj/17AkL5OD6+v8A36XMpA==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
